# Análisis de video y estado del tráfico

Workflow de inferencia de VAAET ML 4.0.0. Combina el pipeline visual compartido con un bundle validado del clasificador de estados y deja la persistencia como una decisión explícita.

In [ ]:
# Environment setup — run once per Colab runtime
import importlib.metadata
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = importlib.util.find_spec("google.colab") is not None
REPO_URL = "https://github.com/zgfnicolas/vaaet.git"
REPO_DIR = Path("/content/vaaet")

if IN_COLAB:
    if (REPO_DIR / ".git").is_dir():
        subprocess.check_call(["git", "-C", str(REPO_DIR), "pull", "--ff-only"])
    else:
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])
    REPO_ROOT = REPO_DIR.resolve()
else:
    candidates = [Path.cwd(), *Path.cwd().parents]
    REPO_ROOT = next(
        (path for path in candidates if (path / "pyproject.toml").is_file() and (path / "src/vaaet").is_dir()),
        None,
    )
    if REPO_ROOT is None:
        raise RuntimeError("VAAET repository root not found")

os.chdir(REPO_ROOT)
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_ROOT}[vision,training,visualization,database]"]
)
subprocess.check_call([sys.executable, "-m", "pip", "check"])

def package_version(name: str) -> str:
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return "not required"

print({name: package_version(name) for name in ("numpy", "tensorflow", "opencv-python-headless", "ultralytics-opencv-headless")})

import os
import shutil

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sqlalchemy import text as sa_text

from vaaet.artifacts import MANIFEST_FILE, create_manifest, validate_manifest
from vaaet.data.database import get_engine, get_optional_db_config, hydrate_db_environment_from_colab
from vaaet.data.persistence import persist_classified_telemetry
from vaaet.features.labeling import assign_traffic_state
from vaaet.inference.traffic_state import classify_raw_telemetry
from vaaet.logging import configure_logging
from vaaet.settings import DRIVE_ARTIFACT_DIR, FEATURE_COLS, LABEL_MAP_PATH, MODEL_DIR, MODEL_PATH, MODEL_VERSION, RANDOM_SEED, SCALER_PATH, STATE_LABELS
from vaaet.vision.analysis import TrafficStatePrediction, analyze_video

hydrate_db_environment_from_colab()
configure_logging()
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)
print(f"Python {sys.version.split()[0]} | NumPy {np.__version__} | TensorFlow {tf.__version__} | GPU {bool(tf.config.list_physical_devices('GPU'))}")
print(f"✅ inference workflow ready | root={REPO_ROOT} | commit={subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip()}")

_model_dir_abs = REPO_ROOT / MODEL_DIR
_model_dir_abs.mkdir(parents=True, exist_ok=True)
_ARTIFACT_NAMES = [Path(MODEL_PATH).name, Path(SCALER_PATH).name, Path(LABEL_MAP_PATH).name, MANIFEST_FILE]

def _bundle_paths(directory: Path) -> dict[str, Path]:
    return {name: directory / name for name in _ARTIFACT_NAMES}

paths = _bundle_paths(_model_dir_abs)
source = "local"
if not all(path.is_file() for path in paths.values()) and IN_COLAB:
    try:
        from google.colab import drive

        drive.mount("/content/drive", force_remount=False)
        drive_paths = _bundle_paths(Path("/content/drive") / DRIVE_ARTIFACT_DIR)
        if all(path.is_file() for path in drive_paths.values()):
            for name, path in drive_paths.items():
                shutil.copy2(path, paths[name])
            source = "Google Drive"
    except Exception as exc:
        print(f"Drive unavailable: {exc}")

if not all(path.is_file() for path in paths.values()) and IN_COLAB:
    from google.colab import files

    print("Upload the complete four-file model bundle:", _ARTIFACT_NAMES)
    for name, content in files.upload().items():
        if name in _ARTIFACT_NAMES:
            paths[name].write_bytes(content)
    source = "upload"

if all(path.is_file() for path in paths.values()):
    validate_manifest(_model_dir_abs)
    model = tf.keras.models.load_model(paths[Path(MODEL_PATH).name])
    scaler = joblib.load(paths[Path(SCALER_PATH).name])
    label_mapping = joblib.load(paths[Path(LABEL_MAP_PATH).name])
    print(f"✅ Valid bundle loaded from {source}")
else:
    missing = [name for name, path in paths.items() if not path.is_file()]
    raise FileNotFoundError(f"Incomplete model bundle; missing: {missing}")


## 1. Seleccionar el clip

In [ ]:
VIDEO_PATH: Path | None = None
if IN_COLAB:
    from google.colab import files

    uploaded = files.upload()
    if uploaded:
        VIDEO_PATH = Path(next(iter(uploaded))).resolve()
else:
    candidate = REPO_ROOT / "data/sample/sample.mp4"
    VIDEO_PATH = candidate if candidate.is_file() else None

print(f"Clip: {VIDEO_PATH or 'not selected'}")

## 2. Análisis anotado e inferencia

In [ ]:
def prediction_provider(raw: pd.DataFrame) -> TrafficStatePrediction | None:
    try:
        classified = classify_raw_telemetry(
            raw,
            model,
            scaler,
            label_mapping=label_mapping,
            feature_cols=FEATURE_COLS,
            model_version=MODEL_VERSION,
            inference_mode="auto",
        )
    except ValueError:
        return None
    if classified.empty:
        return None
    latest = classified.iloc[-1]
    return TrafficStatePrediction(
        state=int(latest["traffic_state"]),
        label=str(latest["state_label"]),
        confidence=float(latest["confidence"]),
        evidence=float(latest.get("accident_evidence_score", 0.0)),
    )

if VIDEO_PATH is None or not VIDEO_PATH.is_file():
    raise FileNotFoundError("Select or upload a valid MP4 clip before continuing")

OUTPUT_VIDEO = Path("/content") / f"{VIDEO_PATH.stem}_vaaet_analyzed.mp4" if IN_COLAB else VIDEO_PATH.with_name(f"{VIDEO_PATH.stem}_vaaet_analyzed.mp4")
analysis_result = analyze_video(VIDEO_PATH, OUTPUT_VIDEO, prediction_provider=prediction_provider)
df_telemetry = analysis_result.telemetry
df_classified = classify_raw_telemetry(
    df_telemetry,
    model,
    scaler,
    label_mapping=label_mapping,
    feature_cols=FEATURE_COLS,
    model_version=MODEL_VERSION,
    inference_mode="auto",
)
display(df_classified)
print(f"✅ Annotated video: {analysis_result.video_path}")

if IN_COLAB:
    from google.colab import files

    files.download(str(analysis_result.video_path))

# Cell 3 — Feature Engineering + Classification
#
# This cell always recomputes classification from df_telemetry using the
# shared vaaet.inference.traffic_state module so that accident gating and contracts stay
# aligned with the training workflow.

try:
    if df_telemetry is not None and not df_telemetry.empty:
        df_classified = classify_raw_telemetry(
            df_telemetry,
            model,
            scaler,
            label_mapping=label_mapping,
            model_version=MODEL_VERSION,
        )
        if df_classified.empty:
            print("⚠️ Insufficient telemetry for classification")
        else:
            print("✅ Classification complete:")
            for code in sorted(df_classified["traffic_state"].unique()):
                count = int((df_classified["traffic_state"] == code).sum())
                label = STATE_LABELS.get(int(code), "Unknown")
                print(f"   {label:>10}: {count} records")
            if "accident_gate_applied" in df_classified.columns:
                gated = int(df_classified["accident_gate_applied"].sum())
                print(f"   Conservative accident gate applied: {gated} record(s)")
    else:
        print("⚠️ No telemetry data — run Cell 2 or Cell 2b first")
        df_classified = None
except NameError:
    print("⚠️ df_telemetry not defined — run Cell 2 or Cell 2b first")
    df_classified = None
except Exception as e:
    print(f"🔴 Classification error: {e}")
    df_classified = None


In [ ]:
# Cell 4 — Persist Results to Database (Optional)

try:
    db_config = get_optional_db_config(interactive=False)
except Exception:
    db_config = None

if db_config is not None:
    try:
        if df_classified is not None and not df_classified.empty:
            persisted = persist_classified_telemetry(
                df_classified,
                config=db_config,
                model_version=MODEL_VERSION,
            )
            print(f"✅ Persistence completed: {persisted.telemetry_rows} telemetry rows | {persisted.classification_rows} classifications")
        else:
            print("⚠️ Run Cells 2-3 first, then re-run this cell")
    except NameError:
        print("⚠️ Run Cells 2-3 first, then re-run this cell")
    except Exception as e:
        print(f"🔴 Persistence error: {e}")
        print("   Classified data is available in-memory (df_classified)")
else:
    print("⚠️ No DB configured — skipping persistence (data stays in-memory as df_classified)")
    print("   Set DB_HOST / DB_NAME / DB_USER / DB_PASSWORD env vars to enable.")


## Feedback Loop — Re-training

This cell implements the self-improvement cycle:
1. Load human-validated records from `traffic_classifications` (where `is_human_validated = TRUE`)
2. Merge with original training data
3. Re-train the MLP with the expanded dataset
4. Export updated `.keras` artifact

This closes the feedback loop: **production → HITL validation → re-training → better production**.

In [ ]:
# Cell 5 — Feedback Loop: Re-train with HITL Data (Optional)

from sklearn.preprocessing import StandardScaler as _StandardScaler
from sklearn.model_selection import train_test_split as _train_test_split
from sklearn.metrics import f1_score as _f1_score
from imblearn.over_sampling import SMOTE as _SMOTE


def retrain_with_feedback(config: dict[str, str]) -> None:
    """Re-train the classifier using human-validated data.

    Loads validated classifications from the database, merges them with
    the original training data, and re-trains the MLP model. The updated
    model is exported only if F1-macro improves over the current one.

    Args:
        config: Database credentials.
    """
    engine = get_engine(config)

    # Load human-validated records
    query = """
        SELECT tr.*, tc.traffic_state AS validated_state
        FROM telemetry_raw tr
        JOIN traffic_classifications tc ON tc.telemetry_id = tr.id
        WHERE tc.is_human_validated = TRUE
        ORDER BY tr.record_time
    """
    df_validated = pd.read_sql(sa_text(query), engine)
    engine.dispose()

    if df_validated.empty:
        print("⚠️ No human-validated records found. Skipping re-training.")
        return

    print(f"📊 Loaded {len(df_validated)} validated records")

    # Merge with original training data 
    csv_path = os.path.join(REPO_ROOT, "data", "processed", "traffic_telemetry.csv")
    if not os.path.exists(csv_path):
        print("🔴 Original training CSV not found. Run the training workflow first.")
        return

    df_original = pd.read_csv(csv_path)
    if "traffic_state" not in df_original.columns:
        df_original["traffic_state"] = assign_traffic_state(df_original)

    # Use validated_state as ground truth for HITL records
    df_validated_features = df_validated[FEATURE_COLS].copy()
    df_validated_features["traffic_state"] = df_validated["validated_state"].astype(int)

    df_combined = pd.concat(
        [df_original[FEATURE_COLS + ["traffic_state"]], df_validated_features],
        ignore_index=True,
    )
    print(f"📊 Combined dataset: {len(df_combined)} records "
          f"({len(df_original)} original + {len(df_validated)} validated)")

    # Split (raw, unscaled) 
    X_raw = df_combined[FEATURE_COLS].values
    y = df_combined["traffic_state"].values

    X_train_raw, X_test_raw, y_train, y_test = _train_test_split(
        X_raw, y, test_size=0.2, stratify=y, random_state=RANDOM_SEED,
    )

    # Evaluate old model on test set (using its own scaler) 
    X_test_old = scaler.transform(X_test_raw)
    y_pred_old = model.predict(X_test_old, verbose=0).argmax(axis=1)
    f1_old = _f1_score(y_test, y_pred_old, average="macro", zero_division=0)

    # Fit new scaler + SMOTE on training data
    new_scaler = _StandardScaler()
    X_train_new = new_scaler.fit_transform(X_train_raw)
    X_test_new = new_scaler.transform(X_test_raw)

    train_counts = np.bincount(y_train)
    min_class = train_counts[train_counts > 0].min()
    k_neighbors = min(5, min_class - 1) if min_class > 1 else 1

    if min_class >= 2:
        sm = _SMOTE(random_state=RANDOM_SEED, k_neighbors=k_neighbors)
        X_train_res, y_train_res = sm.fit_resample(X_train_new, y_train)
        print(f"✅ SMOTE applied (k_neighbors={k_neighbors})")
    else:
        X_train_res, y_train_res = X_train_new, y_train
        print("⚠️ SMOTE skipped — class with <2 samples")

    # Re-train MLP
    from tensorflow.keras.models import Sequential as _Sequential
    from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input
    from tensorflow.keras.callbacks import EarlyStopping

    n_classes = len(np.unique(y))
    new_model = _Sequential([
        Input(shape=(X_train_res.shape[1],)),
        Dense(64, activation="relu"),
        BatchNormalization(),
        Dropout(0.3),
        Dense(32, activation="relu"),
        BatchNormalization(),
        Dropout(0.2),
        Dense(n_classes, activation="softmax"),
    ], name="traffic_state_classifier_retrained")

    new_model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    new_model.fit(
        X_train_res, y_train_res,
        epochs=200,
        batch_size=32,
        validation_split=0.2,
        callbacks=[EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True)],
        verbose=1,
    )

    # Evaluate new model on test set
    y_pred_new = new_model.predict(X_test_new, verbose=0).argmax(axis=1)
    f1_new = _f1_score(y_test, y_pred_new, average="macro", zero_division=0)

    print(f"\n📊 F1-macro comparison:")
    print(f"   Current model: {f1_old:.4f}")
    print(f"   Retrained:     {f1_new:.4f}")

    # Export if improved
    if f1_new > f1_old:
        model_path = os.path.join(REPO_ROOT, "artifacts", "traffic-state", "traffic_classifier.keras")
        scaler_path = os.path.join(REPO_ROOT, "artifacts", "traffic-state", "feature_scaler.joblib")
        label_path = os.path.join(REPO_ROOT, "artifacts", "traffic-state", "label_mapping.joblib")
        new_model.save(model_path)
        joblib.dump(new_scaler, scaler_path)
        joblib.dump(dict(STATE_LABELS), label_path)
        create_manifest(
            os.path.dirname(model_path),
            metrics={"f1_macro": float(f1_new), "previous_f1_macro": float(f1_old)},
            data_provenance={
                "origin": "hitl-feedback-retraining",
                "record_count": int(len(df_combined)),
                "human_validated_record_count": int(len(df_validated)),
                "synthetic_data_included": bool("data_origin" in df_original and (df_original["data_origin"] == "synthetic").any()),
            },
        )
        print(f"✅ Improved model exported → {model_path}")
        print(f"   Complete four-file bundle exported → {os.path.dirname(model_path)}")
    else:
        print("⚠️ Retrained model did not improve. Keeping current model.")


# Execution
# Uncomment when HITL data is available in traffic_classifications:
# retrain_config = get_db_config()
# retrain_with_feedback(retrain_config)
print("⚠️ Re-training requires human-validated data in traffic_classifications")

## Visualization

Summary dashboard showing traffic state distribution, speed timeline, and
classification confidence. Only runs after Cells 2-3 have been executed.

In [ ]:
# Cell 6 — Visualization Dashboard

import matplotlib.pyplot as plt

def show_dashboard(df: pd.DataFrame) -> None:
    """Display a 5-panel summary dashboard for the classified telemetry.

    Panels:
      1. Traffic state distribution (bar)
      2. Average speed over time (line)
      3. Classification confidence histogram
      4. Vehicle counts by type (stacked area)
      5. Speed vs. total vehicles scatter

    Args:
        df: Classified DataFrame with traffic_state, avg_speed, confidence,
            and per-type count columns.
    """
    fig = plt.figure(figsize=(20, 10))
    colors = ["#2ecc71", "#f39c12", "#e74c3c", "#8e44ad"]
    type_colors = {
        "car": "#3498db", "truck": "#e67e22", "bus": "#e74c3c",
        "motorcycle": "#2ecc71", "bicycle": "#9b59b6",
    }

    # Panel 1: State distribution
    ax1 = fig.add_subplot(2, 3, 1)
    dist = df["traffic_state"].value_counts().sort_index()
    state_names = [label_mapping.get(c, f"State {c}") for c in sorted(dist.index)]
    state_colors = [colors[c] for c in sorted(dist.index)]
    ax1.bar(state_names, dist.values, color=state_colors)
    ax1.set_title("Traffic State Distribution")
    ax1.set_ylabel("Records")

    # Panel 2: Speed timeline
    ax2 = fig.add_subplot(2, 3, 2)
    x_axis = range(len(df))
    ax2.plot(x_axis, df["avg_speed"], color="#3498db", linewidth=1.5, label="Avg Speed")
    if "speed_variance" in df.columns:
        ax2.fill_between(
            x_axis,
            df["avg_speed"] - df["speed_variance"].clip(lower=0).pow(0.5),
            df["avg_speed"] + df["speed_variance"].clip(lower=0).pow(0.5),
            alpha=0.2, color="#3498db", label="±1 σ",
        )
    ax2.set_title("Average Speed Over Time")
    ax2.set_xlabel("Minute")
    ax2.set_ylabel("Speed (km/h)")
    ax2.legend(fontsize=8)
    ax2.grid(True, alpha=0.3)

    # Panel 3: Confidence distribution
    ax3 = fig.add_subplot(2, 3, 3)
    ax3.hist(df["confidence"], bins=20, color="#9b59b6", edgecolor="white")
    ax3.axvline(0.8, color="#e74c3c", linestyle="--", linewidth=1, label="Threshold 0.8")
    ax3.set_title("Classification Confidence")
    ax3.set_xlabel("Confidence")
    ax3.set_ylabel("Frequency")
    ax3.legend(fontsize=8)

    # Panel 4: Vehicle counts by type (stacked area)
    ax4 = fig.add_subplot(2, 3, 4)
    count_cols = [c for c in ["count_car", "count_truck", "count_bus",
                               "count_motorcycle", "count_bicycle"]
                  if c in df.columns]
    if count_cols:
        df_counts = df[count_cols].fillna(0)
        ax4.stackplot(
            x_axis, *[df_counts[c] for c in count_cols],
            labels=[c.replace("count_", "") for c in count_cols],
            colors=[type_colors.get(c.replace("count_", ""), "#999") for c in count_cols],
            alpha=0.8,
        )
        ax4.set_title("Vehicle Counts by Type")
        ax4.set_xlabel("Minute")
        ax4.set_ylabel("Count")
        ax4.legend(loc="upper left", fontsize=7)
    else:
        ax4.text(0.5, 0.5, "No count data", ha="center", va="center")
        ax4.set_title("Vehicle Counts by Type")

    # Panel 5: Speed vs. total vehicles (scatter)
    ax5 = fig.add_subplot(2, 3, 5)
    if "total_vehicles" in df.columns:
        scatter_colors = [colors[c] if c < len(colors) else "#999"
                          for c in df["traffic_state"]]
        ax5.scatter(df["total_vehicles"], df["avg_speed"], c=scatter_colors,
                    alpha=0.7, edgecolors="white", linewidth=0.5)
        ax5.set_xlabel("Total Vehicles")
        ax5.set_ylabel("Avg Speed (km/h)")
        ax5.set_title("Speed vs. Volume")
        ax5.grid(True, alpha=0.3)
    else:
        ax5.text(0.5, 0.5, "No volume data", ha="center", va="center")
        ax5.set_title("Speed vs. Volume")

    # Panel 6: Summary text panel
    ax6 = fig.add_subplot(2, 3, 6)
    ax6.axis("off")
    summary_lines = [
        f"📊 Total records: {len(df)}",
        f"⏱️  Avg speed: {df['avg_speed'].mean():.1f} km/h",
        f"📈 Max speed: {df['avg_speed'].max():.1f} km/h",
        f"📉 Min speed: {df['avg_speed'].min():.1f} km/h",
    ]
    if "total_vehicles" in df.columns:
        summary_lines.append(f"🚗 Total vehicles: {df['total_vehicles'].sum():.0f}")
    if "confidence" in df.columns:
        summary_lines.append(f"🎯 Avg confidence: {df['confidence'].mean():.3f}")
        low_conf = (df["confidence"] < 0.8).sum()
        summary_lines.append(f"⚠️  Low confidence (<0.8): {low_conf}")
    summary_text = "\n".join(summary_lines)
    ax6.text(0.1, 0.5, summary_text, fontsize=11, verticalalignment="center",
             fontfamily="monospace", transform=ax6.transAxes)
    ax6.set_title("Summary")

    plt.tight_layout()
    plt.show()


# Execution — requires df_classified from Cell 3
try:
    if df_classified is not None and not df_classified.empty:
        show_dashboard(df_classified)
    else:
        print("⚠️ No classified data — run Cells 2-3 first")
except NameError:
    print("⚠️ df_classified not defined — run Cells 2-3 first")
except Exception as e:
    print(f"🔴 Dashboard error: {e}")